# BIOM 200: Eating, pets, and the oral and nasal microbiomes

Instructors: Rob Knight (robknight@ucsd.edu)

On Monday, October 19, each participant collected six swabs: the left and right anterior nares and the tongue, once before and once after eating. The swabs were extracted, amplified for the 16S rRNA V4 region, sequenced the same day, and processed in Qiita (study [16596](https://qiita.ucsd.edu/study/description/16596), alongside Study 1). The class designed this study to test two hypotheses:

1. **Eating** changes the oral microbiome but not the nasal microbiome.
2. **Pets** affect the nasal microbiome but not the oral microbiome.

The design also proposed comparing vegetarians with omnivores; see below for why that comparison is not possible here.

The methods were introduced in the first two notebooks, so this one moves quickly through the overview and spends its time on the hypotheses. Before running anything, write down what you expect for each hypothesis and why.

## Two neighbouring sites

The mouth and the nose are a few centimetres apart but host very different communities. The tongue is dominated by *Streptococcus*, *Veillonella*, *Prevotella*, *Neisseria*, *Haemophilus*, and *Rothia*; the anterior nares, as we saw in Study 1, by *Corynebacterium* and *Staphylococcus*. Body site explains more of the variation in the human microbiome than anything else, yet communities also shift from day to day with diet and routine while each person stays recognisable. People who share a home share part of their microbiota with one another and with their dogs, most clearly on the skin.

**Read more:** [Human Microbiome Project Consortium 2012, *Nature*](https://doi.org/10.1038/nature11234) · [Costello et al. 2009, *Science*](https://doi.org/10.1126/science.1177486) · [David et al. 2014, *Genome Biol*](https://doi.org/10.1186/gb-2014-15-7-r89) · [Song et al. 2013, *eLife*](https://doi.org/10.7554/eLife.00458)

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from scripts.biom200 import *   # autoreload picks up edits to the module without a restart

## Load the data

The Study 2 data (Qiita study 16596) are included in `data/class2/`. Participants are identified by the same codes as in Study 1 (P01–P19). Two columns are derived on loading: `site` (tongue or nostril) and a readable version of `pets`. P14 contributed one extra tongue swab, taken after coffee rather than as part of the before/after design; it appears in the overview and in the personal readout but not in the hypothesis tests.

In [2]:
TABLE = "data/class2/16596_feature-table.biom"
METADATA = "data/class2/16596_sample_information.tsv"
TAXONOMY = "data/class2/16596_taxonomy.tsv"
TREE = "data/class2/16596_tree.nwk"

study2 = load_dataset("study2", TABLE, METADATA, TAXONOMY, TREE, group="site", subject="host_subject_id",
                      control_col="sample_type", control_values=["control blank"])
add_metadata_column(study2, "site", lambda m: np.where(m["side"].isin(["left", "right"]), "nostril", "tongue"))
add_metadata_column(study2, "pets", lambda m: m["pets"].astype(str).str.upper().map({"TRUE": "pets", "FALSE": "no pets"}).fillna("not recorded"))

study2: 88 samples and 1427 features; 88 samples have metadata
  4 samples in the metadata have no reads in the table
  7 controls (control blank): median 35 reads vs 11,228 in samples
  81 biological samples retained; fields available for grouping: ['description', 'eating_stage', 'empo_4', 'env_material', 'env_package', 'host_body_habitat', 'host_body_site', 'pets', 'sample_type', 'scientific_name', 'side']
site
nostril    52
tongue     29
pets
no pets         52
pets            28
not recorded     1


Pets and diet are properties of a person, not of a sample, so count people rather than swabs. These counts set the real sample size for Hypothesis 2.

In [3]:
participants = study2.meta.drop_duplicates("host_subject_id")
pd.crosstab(participants["pets"], participants["diet"], margins=True)

diet,omnivore,All
pets,,
no pets,9,9
pets,5,5
All,14,14


## Controls

As in Study 1, check the blanks before trusting the samples: how many reads do they carry, and do they look like reagent background or like a neighbouring sample? A tongue swab carries far more bacterial biomass than a nasal swab, so a tongue well next to a blank is a more likely source of cross-contamination than anything in Study 1.

**Read more:** [Salter et al. 2014, *BMC Biol*](https://doi.org/10.1186/s12915-014-0087-z)

In [4]:
explore_controls(study2)

interactive(children=(Dropdown(description='Level', index=4, options=('Phylum', 'Class', 'Order', 'Family', 'G…

In [5]:
control_report(study2).head(10)

,control,reads,top taxa
0,16596.BLANK.BIOM200.1.A7,125,"Burkholderia ubonensis (26%), Acidovorax_F sol..."
1,16596.BLANK.BIOM200.1.F7,107,"Acidovorax_F soli (24%), Burkholderia ubonensi..."
2,16596.BLANK.BIOM200.1.D1,95,"Burkholderia ubonensis (57%), Curvibacter lanc..."
3,16596.BLANK.BIOM200.1.F9,35,"Burkholderia ubonensis (34%), Methylophilaceae..."
4,16596.BLANK.BIOM200.1.D2,23,"Burkholderia ubonensis (30%), Acidovorax_F sol..."
5,16596.BLANK.BIOM200.1.E9,14,"Corynebacterium (43%), Burkholderia ubonensis ..."
6,16596.BLANK.BIOM200.1.E7,6,"Burkholderia ubonensis (67%), Curvibacter lanc..."


## Overview: composition, depth, and diversity

A quick look at the whole study before testing anything. Group the bar plot by `site`, and tick **Group means** to compare the average tongue and nostril communities.

In [6]:
add_composition(study2)
explore_composition(study2)

interactive(children=(Dropdown(description='Level', index=4, options=('Phylum', 'Class', 'Order', 'Family', 'G…

Choose a rarefaction depth. Tongue swabs usually yield more reads than nasal swabs, so check which site loses samples at the depth you pick: a person who loses one swab drops out of the paired comparisons below.

In [7]:
add_curves(study2)
explore_rarefaction(study2)

interactive(children=(Dropdown(description='Colour by', options=('site', 'description', 'eating_stage', 'empo_…

In [8]:
add_diversity(study2)

Rarefying to 600 reads; 9 of 81 samples dropped


/opt/homebrew/Caskroom/miniforge/base/envs/rachis-qiime2-2026.7/lib/python3.12/site-packages/skbio/util/_warning.py:42: DeprecationWarning: `observed_otus` was renamed to `sobs` in 0.6.0. The old name is kept as an alias but is deprecated.
  warn(message, warning)


Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

faithpd -i /var/folders/9k/dg_fpv9j08dd5_7mzskwwbfh0000gn/T/qiime2/lucas/processes/17403-1792509726.5898@lucas/tmp/rachis-OutPath-iobt2k2j -t /var/folders/9k/dg_fpv9j08dd5_7mzskwwbfh0000gn/T/qiime2/lucas/data/34fcdc27-4495-4ce8-9f39-2fffcd50a752/data/tree.nwk -o /var/folders/9k/dg_fpv9j08dd5_7mzskwwbfh0000gn/T/qiime2/lucas/processes/17403-1792509726.5898@lucas/tmp/rachis-OutPath-lxh_2syc

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

ssu -i /var/folders/9k/dg_fpv9j08dd5_7mzskwwbfh0000gn/T/qiime2/lucas/data/e6fa576a-d26e-4714-a91b-21eb419c622b/data/feature-table.biom -t /var

observed_features        shannon       faith_pd        
                     mean    std    mean   std     mean     std
site                                                           
nostril             35.28  15.79    2.82  0.63  2634.37  729.24
tongue              47.21  13.95    3.39  0.76  3209.79  493.12

Split the study by site, keeping only the samples from the before/after design. Both halves keep the same rarefied table, so their diversity values are directly comparable with the whole-study values.

In [9]:
design = study2.meta["eating_stage"].notna()   # excludes P14's swab after coffee
tongue = subset_dataset(study2, "study2-tongue", design & (study2.meta["site"] == "tongue"), group="eating_stage")
nostril = subset_dataset(study2, "study2-nostril", design & (study2.meta["site"] == "nostril"), group="eating_stage")

study2-tongue: 28 samples, 28 after rarefaction
study2-nostril: 52 samples, 43 after rarefaction


In [10]:
top_taxa(tongue, "Genus")

,mean relative abundance,samples carrying it,host_subject_ids carrying it,dominant in (of 14 host_subject_ids)
Genus,,,,
Haemophilus_D,26.8%,100.0%,100.0%,5
Prevotella,18.1%,100.0%,100.0%,4
Neisseria,14.0%,100.0%,100.0%,3
Streptococcus,8.6%,100.0%,100.0%,2
Veillonella_A,6.6%,100.0%,100.0%,0
Fusobacterium_C,4.3%,100.0%,100.0%,0
Porphyromonas_A,3.9%,100.0%,100.0%,0
Caldora,3.0%,96.4%,100.0%,0
Granulicatella,2.0%,100.0%,100.0%,0


In [11]:
top_taxa(nostril, "Genus")

,mean relative abundance,samples carrying it,host_subject_ids carrying it,dominant in (of 14 host_subject_ids)
Genus,,,,
Corynebacterium,41.0%,100.0%,100.0%,10
Staphylococcus,14.7%,96.2%,100.0%,1
Lawsonella,13.5%,96.2%,100.0%,1
Neisseriaceae (f.),4.5%,76.9%,100.0%,1
Dolosigranulum,4.1%,80.8%,100.0%,0
Anaerococcus,3.0%,76.9%,92.9%,0
Lactobacillus,2.5%,32.7%,50.0%,1
Burkholderia,1.8%,80.8%,100.0%,0
Haemophilus_D,1.8%,94.2%,100.0%,0


Every participant contributed both tongue and nostril swabs, so the site comparison is paired: each line joins one person's mean tongue and mean nostril diversity. The PCoA shows the same contrast for whole communities.

In [12]:
explore_paired_alpha(study2, group="site", pair_by="host_subject_id")

interactive(children=(Dropdown(description='Metric', index=1, options=('observed_features', 'shannon', 'faith_…

In [13]:
explore_pcoa(study2)

interactive(children=(Dropdown(description='Metric', options=('bray_curtis', 'jaccard', 'unweighted_unifrac', …

**What we see.** The tongue and the nostrils host entirely different communities. The PCoA separates the two sites cleanly (PERMANOVA on Bray–Curtis, pseudo-F ≈ 44, p = 0.001), and the random forest below classifies every sample correctly. The tongue is dominated by *Haemophilus*, *Prevotella*, *Neisseria*, and *Streptococcus*, each found in every tongue swab; the nostrils by *Corynebacterium* (41% of reads on average and the most abundant genus in 10 of 14 people), *Staphylococcus*, and *Lawsonella*, much as in Study 1. Tongue swabs are usually, but not always, the more diverse: Shannon diversity was higher on the tongue in 9 of 14 people (paired Wilcoxon p ≈ 0.06–0.09, depending on the rarefaction draw). At the default depth of 600 reads, 9 of 81 samples drop out, all of them nostril swabs.

How large is the difference between sites compared with the differences between people within a site? Keep that scale in mind: any effect of eating or pets is likely to be much smaller.

## Can a classifier tell tongue from nostril?

A random forest learns rules that predict a label from the relative abundance of each taxon. To estimate how well it would do on new people, it is trained on most participants and tested on the rest, five times over, so the model is never tested on anyone whose samples it saw in training. Compare the accuracy with the baseline of always guessing the commonest label.

Tongue vs. nostril should be easy, given what the overview showed. Then switch **Predict** to `eating_stage` or `pets`: how much harder are those, and which taxa does the model rely on?

In [14]:
explore_classifier(study2, target="site")

interactive(children=(Dropdown(description='Predict', options=('site', 'description', 'eating_stage', 'empo_4'…

## Hypothesis 1: eating changes the oral but not the nasal microbiome

Each person was swabbed before and after eating, so the comparison is **paired**: each person is compared with themselves. Start with diversity at each site. For the nostrils, each point is the mean of the left and right swabs.

In [15]:
explore_paired_alpha(tongue, group="eating_stage", pair_by="host_subject_id")

interactive(children=(Dropdown(description='Metric', index=1, options=('observed_features', 'shannon', 'faith_…

In [16]:
explore_paired_alpha(nostril, group="eating_stage", pair_by="host_subject_id")

interactive(children=(Dropdown(description='Metric', index=1, options=('observed_features', 'shannon', 'faith_…

Diversity can stay the same while the community changes, so now measure how far each person's community moved between the two time points. The hypothesis predicts a larger move on the tongue than in the nostrils.

A difference between two swabs is not all biology: two swabs of the same site taken minutes apart also differ, through sampling, extraction, and sequencing. The left and right nostrils swabbed at the same time give a reference for that noise in the nose. If the nostrils change no more after eating than the two nostrils differ from each other, eating had no detectable effect on them. The tongue has no such reference in this design; a duplicate tongue swab at each time point would have provided one.

Each point is one person; grey lines join the same person across comparisons, and the table below the plot gives paired Wilcoxon tests.

In [17]:
explore_change(study2, change="eating_stage", split="site", baseline="side")

interactive(children=(Dropdown(description='Metric', options=('bray_curtis', 'jaccard', 'unweighted_unifrac', …

The same question in the ordinations: grey lines join each person's samples. Short lines mean little change.

In [18]:
explore_pcoa(tongue, connect="host_subject_id")

interactive(children=(Dropdown(description='Metric', options=('bray_curtis', 'jaccard', 'unweighted_unifrac', …

In [19]:
explore_pcoa(nostril, connect="host_subject_id")

interactive(children=(Dropdown(description='Metric', options=('bray_curtis', 'jaccard', 'unweighted_unifrac', …

Which genera changed? ANCOM-BC compares after with before, separately at each site. These tests treat every sample as independent. Adjusting for participant would compare each person with themselves, but with only a few samples per person that model has too little data left to estimate variability and reports many false positives (in checks with shuffled labels, most runs found "significant" genera where there were none); the paired plots above handle the pairing instead. Positive log2 fold changes are higher after eating.

Food carries DNA of its own: plant plastids (chloroplasts and their relatives in non-green tissues such as seeds) and mitochondria contain 16S genes, and fermented foods contain bacteria such as *Lactococcus* and *Lactobacillus*. Plastids descend from cyanobacteria, and Greengenes2 classifies their sequences within the Cyanobacteria, so plant DNA from a meal can appear under a cyanobacterial genus name such as *Caldora*. For any genus that rises after eating, ask whether it could have come from the meal rather than from a change in the resident community. And because 16S data are relative, a large influx of food DNA lowers the share of everything else even if nothing else changed.

**Read more:** [Lin & Peddada 2020, *Nat Commun*](https://doi.org/10.1038/s41467-020-17041-7)

In [20]:
add_differential_abundance(tongue, group="eating_stage", reference="pre_eating")
explore_forest(tongue, group="eating_stage")

Testing 91 of 489 genera for eating_stage
2 genus–comparison pairs with q < 0.05 (Holm-adjusted); full table in results/study2-tongue/ancombc-eating_stage.tsv


interactive(children=(Dropdown(description='Comparison', options=('post_eating vs pre_eating',), style=Descrip…

Which sequences are behind the significant genera? Each entry below is one ASV: the genus it was assigned to, its mean share before and after eating, and its full sequence. Paste a sequence into [NCBI BLAST](https://blast.ncbi.nlm.nih.gov/Blast.cgi) (blastn, nucleotide collection) to see what it matches best.

In [21]:
significant_asvs(tongue, group="eating_stage")

>Caldora | post_eating 5.84% | pre_eating 0.09%
TACAGAGGATGCAAGCGTTATCCGGAATGATTGGGCGTAAAGCGTCTGTAGGTGGCTTTTTAAGTCCGCCGTCAAATCCCAGGGCTCAACCCTGGACAGGCGGTGGAAACTACCAAGCTGGAGTACGGTAGGGGCAGAGGGAATTTCCGG
>Latescibacterota (p.) | post_eating 0.65% | pre_eating 0.01%
GACGGGGGGGGCAAGTGTTCTTCGGAATGACTAGGCGTAAAGGGCACGTAGGCGGTGAATCGGGTTGAAAGCGAAAGTCGCCAAAAACTGGTGGAATGCTCTCGAAACCAATTCACTTGAGTGAGACAGAGGAGAGTGGAATTTCGTGTG


,genus,taxonomy,post_eating,pre_eating
ASV sequence,,,,
TACAGAGGATGCAAGCGTTATCCGGAATGATTGGGCGTAAAGCGTCTGTAGGTGGCTTTTTAAGTCCGCCGTCAAATCCCAGGGCTCAACCCTGGACAGGCGGTGGAAACTACCAAGCTGGAGTACGGTAGGGGCAGAGGGAATTTCCGG,Caldora,d__Bacteria; p__Cyanobacteriota; c__Cyanobacte...,0.058381,0.000943
GACGGGGGGGGCAAGTGTTCTTCGGAATGACTAGGCGTAAAGGGCACGTAGGCGGTGAATCGGGTTGAAAGCGAAAGTCGCCAAAAACTGGTGGAATGCTCTCGAAACCAATTCACTTGAGTGAGACAGAGGAGAGTGGAATTTCGTGTG,Latescibacterota (p.),d__Bacteria; p__Latescibacterota; c__; o__; f_...,0.006494,0.000106


In [23]:
add_differential_abundance(nostril, group="eating_stage", reference="pre_eating")
explore_forest(nostril, group="eating_stage")

Testing 43 of 489 genera for eating_stage
0 genus–comparison pairs with q < 0.05 (Holm-adjusted); full table in results/study2-nostril/ancombc-eating_stage.tsv


interactive(children=(Dropdown(description='Comparison', options=('post_eating vs pre_eating',), style=Descrip…

**What we see.** Eating changed neither site's diversity: Shannon diversity rose slightly on the tongue after eating, but not significantly (paired Wilcoxon p ≈ 0.2), and did not change in the nostrils. The communities as a whole moved little. Each person's tongue moved about as far between the two swabs as their nostrils did (median Bray–Curtis 0.17 vs. 0.16, p ≈ 0.7), and the nostrils changed less after eating than the left and right nostrils differ at the same moment (median 0.16 vs. 0.28; smaller in 10 of 11 people). In the nose, eating left no trace.

The one clear signal is on the tongue: two sequences rose after eating in all 14 participants. One is assigned to the cyanobacterial genus *Caldora* (from about 0.1% of reads before eating to 6% on average after, and over 20% in two people); the other is unclassified below the phylum Latescibacterota. Neither appears in the nostrils or the blanks. BLAST identifies both as DNA from the meal, which included nuts: the *Caldora* sequence is a nut plastid (not a chloroplast), and the Latescibacterota sequence is the almond mitochondrion. Neither is a bacterium, and neither lives on the tongue. The hypothesis holds in a narrow sense, since only the tongue changed, but what changed is what was eaten, not the resident community.

## Vegetarians vs. omnivores

The design proposed comparing vegetarians with omnivores, but every participant reported an omnivorous diet (see the table of participants above), so this question cannot be answered with these data. No analysis can recover a comparison that the sampling did not include.

This is a common and avoidable failure in study design: before collecting samples, check that every group you plan to compare is represented, and by enough people to detect the effect you expect. How many vegetarians would you have needed, and how would you have recruited them?

## Hypothesis 2: pets affect the nasal but not the oral microbiome

Pet ownership is a property of the person, so the unit of replication is the participant, not the swab. Each person contributed four nostril swabs and two tongue swabs; treating them as independent would count each person several times and overstate the evidence. The diversity plots therefore show one point per participant: the mean of their swabs at that site.

**Read more:** [Song et al. 2013, *eLife*](https://doi.org/10.7554/eLife.00458)

In [24]:
explore_alpha(tongue, group="pets", average_by="host_subject_id")

interactive(children=(Dropdown(description='Metric', index=1, options=('observed_features', 'shannon', 'faith_…

In [25]:
explore_alpha(nostril, group="pets", average_by="host_subject_id")

interactive(children=(Dropdown(description='Metric', index=1, options=('observed_features', 'shannon', 'faith_…

In [26]:
explore_pcoa(nostril, color_by="pets", connect="host_subject_id")

interactive(children=(Dropdown(description='Metric', options=('bray_curtis', 'jaccard', 'unweighted_unifrac', …

For a test of whole communities that counts each person once, keep a single swab per person: the tongue swab and the left nostril swab taken before eating. With so few people in each group, a non-significant result says little about whether pets matter. Note also that the hypothesis compares two sites: finding an effect in the nostrils but not on the tongue does not by itself show that pets affect the two sites differently, because a non-significant result at one site may simply reflect too few people.

In [27]:
tongue_one = subset_dataset(tongue, "study2-tongue-pre", tongue.meta["eating_stage"] == "pre_eating", group="pets")
nostril_one = subset_dataset(nostril, "study2-nostril-pre-left",
                             (nostril.meta["eating_stage"] == "pre_eating") & (nostril.meta["side"] == "left"), group="pets")
pd.concat({"tongue": permanova_table(tongue_one, ["pets"]), "nostril": permanova_table(nostril_one, ["pets"])})

study2-tongue-pre: 14 samples, 14 after rarefaction
study2-nostril-pre-left: 14 samples, 12 after rarefaction


metric grouping  pseudo-F      p
tongue  0         bray_curtis     pets      1.60  0.159
        1             jaccard     pets      1.08  0.280
        2  unweighted_unifrac     pets      0.76  0.723
        3    weighted_unifrac     pets      1.11  0.317
nostril 0         bray_curtis     pets      0.87  0.462
        1             jaccard     pets      1.00  0.427
        2  unweighted_unifrac     pets      1.49  0.099
        3    weighted_unifrac     pets      0.76  0.568

In [28]:
add_differential_abundance(nostril_one, group="pets", reference="no pets")
explore_forest(nostril_one, group="pets")

Testing 45 of 489 genera for pets
0 genus–comparison pairs with q < 0.05 (Holm-adjusted); full table in results/study2-nostril-pre-left/ancombc-pets.tsv


interactive(children=(Dropdown(description='Comparison', options=('pets vs no pets',), style=DescriptionStyle(…

In [29]:
add_differential_abundance(tongue_one, group="pets", reference="no pets")
explore_forest(tongue_one, group="pets")

Testing 82 of 489 genera for pets
2 genus–comparison pairs with q < 0.05 (Holm-adjusted); full table in results/study2-tongue-pre/ancombc-pets.tsv


interactive(children=(Dropdown(description='Comparison', options=('pets vs no pets',), style=DescriptionStyle(…

**What we see.** We found no evidence that pets affect either site. Pet owners and non-owners had similar diversity at both sites (Mann–Whitney p > 0.3), and with one swab per person, PERMANOVA found no difference in community composition (Bray–Curtis p ≈ 0.1 for the tongue, ≈ 0.5 for the nostrils). The random forest could not predict pet ownership better than always guessing "no pets". ANCOM-BC flagged two tongue genera as less abundant in pet owners, *Kingella* and *Cardiobacterium*, both normal members of the oral flora, and nothing in the nostrils; with five pet owners, and no account taken of anything else that differs between owners and non-owners, treat these as leads, not findings. With 5 vs. 9 people this study could only have detected a large effect, so the absence of a difference is not evidence that pets do not matter.

## Are our noses stable? Study 1 vs. Study 2

Most participants also took part in Study 1, four days earlier, and kept the same participant code. Joining the two studies by `host_subject_id` asks whether a person's nostrils still resemble their own Study 1 nostrils more than anyone else's. Only nostril swabs from people in both studies are kept, and both studies are rarefied to the same depth.

The two studies were sequenced in separate runs, which can shift every sample a little. That affects everyone equally, so compare distances between a person's own samples across the two studies with distances between different people across the two studies, not raw distances between the studies. The tree is not used, because each study comes with its own.

In [30]:
both = combine_studies("study1-2", {
    "Study 1": ("data/class/16596_feature-table.biom", "data/class/16596_sample_information.tsv", "data/class/16596_taxonomy.tsv"),
    "Study 2": (TABLE, METADATA, TAXONOMY)}, group="study", subject="host_subject_id")
nostril_swab = both.meta["side"].isin(["left", "right"])
in_both = both.meta[nostril_swab].groupby("host_subject_id")["study"].transform("nunique") == 2   # nostrils in both studies
both = subset_dataset(both, "study1-2-nostril", in_both.reindex(both.meta.index, fill_value=False))
add_composition(both)
add_curves(both)
add_diversity(both, depth=1000)   # the same depth for both studies; nostril swabs are shallow

study1-2: 179 samples and 2639 features; 179 samples have metadata
  9 samples in the metadata have no reads in the table
  63 controls (control blank): median 403 reads vs 17,168 in samples
  116 biological samples retained; fields available for grouping: ['study', 'collection_date', 'empo_4', 'env_material', 'env_package', 'host_body_habitat', 'host_body_site', 'host_taxid', 'sample_type', 'scientific_name', 'side']
study1-2-nostril: 73 samples
Rarefying to 1,000 reads; 11 of 73 samples dropped


observed_features        shannon      
                     mean    std    mean   std
study                                         
Study 1             42.48  13.99    2.89  0.79
Study 2             44.38  18.68    2.83  0.65

In [31]:
explore_within_between(both, by="host_subject_id", across="study")

interactive(children=(Dropdown(description='Metric', options=('bray_curtis', 'jaccard'), style=DescriptionStyl…

In [32]:
explore_pcoa(both, connect="host_subject_id", color_by="study")

interactive(children=(Dropdown(description='Metric', options=('bray_curtis', 'jaccard'), style=DescriptionStyl…

**What we see.** Our noses are stable over four days. A person's Study 2 nostrils were much closer to their own Study 1 nostrils than to anyone else's (median Bray–Curtis 0.27 vs. 0.64; one-sided Mann–Whitney p far below 0.001), and this held for every participant with samples from both studies at this depth. For scale, two swabs of the same nostril an hour apart in Study 2 differed by about 0.16, and the left and right nostrils on the same day by about 0.28: over four days, a person's nasal community changes about as much as it differs between their two nostrils, and far less than it differs from anyone else's.

## Your own microbiome

Set `PARTICIPANT` to the participant code you were given (ask an instructor if you do not know it), and optionally `COMPARE` to a second code to compare yourself with someone else. The dropdowns change both without rerunning.

The readout shows your tongue and nostril samples next to the study mean for each site, where your samples sit in the PCoA, and a table with, for each sample: its read count; its diversity, as a percentile among all samples from the same site; how far it is from your other samples at that site and from everyone else's; and the most similar sample from another person. Remember what 16S can and cannot tell you: relative abundance of bacterial groups, not counts, function, or health. Nothing here is diagnostic.

In [ ]:
PARTICIPANT = "P01"   # your participant code
COMPARE = None        # optional: a second participant code, e.g. "P02"

explore_participant(study2, participant=PARTICIPANT, compare=COMPARE, by="site", label_by=["site", "side", "eating_stage"])

## Reflection

1. What did you predict for each hypothesis, and what did you find?
2. For Hypothesis 1, how did the left–right reference change your reading of the nasal result? What would duplicate tongue swabs have added?
3. For Hypothesis 2, how many participants did each group contain, and what effect size could this study have detected?
4. Besides owning a pet, how else might pet owners differ from non-owners?
5. If you ran Study 2 again, what would you change: timing of the second swab, meals, replicates, number of people, or metadata?

**Read more:** [Knight et al. 2018, *Nat Rev Microbiol*](https://doi.org/10.1038/s41579-018-0029-9)